# Telco Customer Churn — Exploratory Data Analysis & Feature Engineering

Ce notebook présente la démarche complète d'exploration, de nettoyage ciblé et de préparation des données du dataset **Telco Customer Churn** avant entraînement de modèles de Machine Learning. Il couvre le diagnostic des types, l'imputation logique, l'encodage hybride (ordinal vs nominal) et la mise à l'échelle stratifiée.

## 1. Chargement et diagnostic initial

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_URL)
print(f"Dimensions du jeu de données : {df.shape}")
df.info()

Dimensions du jeu de données : (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  

## 2. Correction des types et traitement des valeurs manquantes

`TotalCharges` est initialement importé comme une chaîne de caractères (`object`). Nous forçons la conversion en float et appliquons une imputation contextuelle (remplacement par `0` pour les comptes récemment créés avec `tenure = 0`).

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Valeurs manquantes détectées dans TotalCharges : {df['TotalCharges'].isna().sum()}")

# Imputation par 0 justifiée par tenure = 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(f"Valeurs manquantes post-imputation : {df['TotalCharges'].isna().sum()}")

Valeurs manquantes détectées dans TotalCharges : 11
Valeurs manquantes post-imputation : 0


## 3. Analyse des valeurs aberrantes (Outliers)

Contrôle de la distribution de `MonthlyCharges` par la méthode de l'écart interquartile (IQR).

In [ ]:
Q1 = df['MonthlyCharges'].quantile(0.25)
Q3 = df['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
borne_basse = Q1 - 1.5 * IQR
borne_haute = Q3 + 1.5 * IQR

outliers = df[(df['MonthlyCharges'] < borne_basse) | (df['MonthlyCharges'] > borne_haute)]
print(f"Intervalle de distribution normale : [{borne_basse:.1f} ; {borne_haute:.1f}]")
print(f"Nombre d'outliers identifiés : {len(outliers)}")

Intervalle de distribution normale : [-46.0 ; 171.4]
Nombre d'outliers identifiés : 0


## 4. Encodage des variables catégorielles

- **Variables binaires :** conversion `Yes`/`No` en `1`/`0`.
- **Variables ordinales :** mapping hiérarchique pour `Contract` reflétant la durée d'engagement.
- **Variables nominales :** One-Hot Encoding pour les modes de paiement afin d'éviter d'introduire un ordre artificiel.

In [ ]:
# Mapping binaire
colonnes_binaires = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in colonnes_binaires:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Encodage ordinal
ordre_contract = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
df['Contract_encoded'] = df['Contract'].map(ordre_contract)

# One-Hot Encoding
df = pd.get_dummies(df, columns=['PaymentMethod'], prefix='Payment', drop_first=True)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,Contract_encoded,Payment_Credit card (automatic),Payment_Electronic check,Payment_Mailed check
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,No,...,No,Month-to-month,1,29.85,29.85,0,0,False,True,False
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,Yes,...,No,One year,0,56.95,1889.50,0,1,False,False,True
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,Yes,...,No,Month-to-month,1,53.85,108.15,1,0,False,False,True
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,Yes,...,No,One year,0,42.30,1840.75,0,1,False,False,False
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,No,...,No,Month-to-month,1,70.70,151.65,1,0,False,True,False


## 5. Feature Engineering : Segmentation temporelle

Discrétisation de la variable continue `tenure` (durée d'abonnement en mois) en paliers opérationnels : *Nouveau*, *Etabli*, *Fidele*.

In [ ]:
def categoriser_tenure(mois):
    if mois <= 12:
        return 'Nouveau'
    elif mois <= 48:
        return 'Etabli'
    return 'Fidele'

df['tenure_categorie'] = df['tenure'].apply(categoriser_tenure)
df['tenure_categorie'].value_counts()

,count
tenure_categorie,
Etabli,2618
Fidele,2239
Nouveau,2186


## 6. Normalisation et découpage stratifié (Train / Test Split)

Le taux d'attrition étant déséquilibré (~26,5 %), nous appliquons un échantillonnage stratifié pour garantir une représentativité identique sur les ensembles d'entraînement et de validation.

In [ ]:
X = df.drop(columns=['Churn', 'customerID', 'Contract', 'tenure_categorie'])
X = pd.get_dummies(X, drop_first=True)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
cols_numeriques = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_train[cols_numeriques] = scaler.fit_transform(X_train[cols_numeriques])
X_test[cols_numeriques] = scaler.transform(X_test[cols_numeriques])

print(f"Échantillon Train : {X_train.shape[0]} lignes | Ratio Churn : {y_train.mean():.3f}")
print(f"Échantillon Test  : {X_test.shape[0]} lignes | Ratio Churn : {y_test.mean():.3f}")

Échantillon Train : 5634 lignes | Ratio Churn : 0.265
Échantillon Test  : 1409 lignes | Ratio Churn : 0.265
